# Baltic Situation Awareness (Lagebild)

NATO C2-oriented **Lagebild** demo: `defense.lagebild` → dataset summary → LLM report → PDF → 24h/48h comparison.

Architecture: see Project docs `situation-awareness-architecture.md`.

**Offline:** set `LAGEBILD_USE_SAMPLE=1` to use `cai-workbench/data/lagebild/sample_lagebild.csv`.

In [ ]:
import os
from pathlib import Path

# Sanitize API key (project env vars often get a trailing space when pasted)
_raw_key = os.environ.get("LAGEBILD_API_KEY")
if _raw_key:
    _k = _raw_key.strip().strip("\ufeff")
    if _k.lower().startswith("bearer "):
        _k = _k[7:].strip()
    os.environ["LAGEBILD_API_KEY"] = _k

# Sample CSV only when Trino/CML is not configured
if not os.environ.get("LAGEBILD_TRINO_HOST") and not os.environ.get(
    "LAGEBILD_CONNECTION_NAME"
):
    os.environ.setdefault("LAGEBILD_USE_SAMPLE", "1")

import sys
ROOT = Path.cwd()
if ROOT.name != "situation-awareness":
    ROOT = ROOT / "cai-workbench" / "situation-awareness"
sys.path.insert(0, str(ROOT))

from src.config import Settings
from src.data_access import load_lagebild_dataframe, dataframe_to_markdown_bounded, lagebild_dataset_summary
from lagebild import run_pipeline

settings = Settings()
settings.artifacts_dir = (ROOT / "artifacts" / "reports").resolve()
print("Artifacts:", settings.artifacts_dir)

## 1. Load data

In [ ]:
df, source = load_lagebild_dataframe(settings, ROOT)
md, df_used = dataframe_to_markdown_bounded(df, settings.max_char_limit)
summary = lagebild_dataset_summary(df_used)
print("Source:", source)
print("Summary:", summary)
print("Rows:", len(df_used), "| Markdown chars:", len(md))
display(df_used.head(10))

## 2. Full pipeline (LLM)

Requires `LAGEBILD_LLM_BASE_URL` and API key/JWT. Set `SKIP_LLM=True` to generate a PDF shell only.

In [ ]:
settings = Settings()  # re-read env after edits above
settings.artifacts_dir = (ROOT / "artifacts" / "reports").resolve()

SKIP_LLM = not os.environ.get("LAGEBILD_API_KEY") and not settings.jwt_path.is_file()

if SKIP_LLM:
    result = run_pipeline(settings, skip_llm=True)
    print("(Skipped LLM — configure model endpoint to run live.)")
else:
    result = run_pipeline(settings)

print(result.dataset_summary)
print("PDF:", result.pdf_path)

## 3. Report text

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result.report_text[:8000]))

## 4. 24h / 48h comparison

In [ ]:
if result.comparison_text:
    display(Markdown(result.comparison_text))
else:
    print("No comparison section (first run or LLM skipped).")

## 5. PDF artifact

Download link and inline preview below (Jupyter cannot open local paths as URLs).

In [ ]:
if result.pdf_path and result.pdf_path.is_file():
    pdf = result.pdf_path
    print(pdf)
    print(f"PDF written ({pdf.stat().st_size:,} bytes)")
    from IPython.display import FileLink, IFrame, Image, display

    if getattr(result, "map_path", None) and result.map_path.is_file():
        display(Image(filename=str(result.map_path)))

    display(FileLink(pdf, result=True))
    try:
        import base64

        b64 = base64.b64encode(pdf.read_bytes()).decode("ascii")
        display(IFrame(src=f"data:application/pdf;base64,{b64}", width=700, height=900))
    except Exception as exc:
        print(f"Inline preview unavailable: {exc}")
else:
    print("No PDF path on result.")